# Module 3: Baseline Models
N-Gram LM and LSTM for next word prediction.

In [ ]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
print('TensorFlow version:', tf.__version__)

## 1. Load Data

In [ ]:
X = np.load('data/lstm_X.npy')
y = np.load('data/lstm_y.npy')

with open('data/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

total_words = len(tokenizer.word_index) + 1
max_sequence_len = X.shape[1] + 1

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Vocabulary size: {total_words} words')
print(f'Sequence length: {max_sequence_len}')

y_categorical = to_categorical(y, num_classes=total_words)
print(f'y_categorical shape: {y_categorical.shape}')

## 2. LSTM Training

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Build the model to display the summary
model.build(input_shape=(None, max_sequence_len-1))
model.summary()

In [ ]:
print('Starting LSTM training...')
history = model.fit(X, y_categorical, epochs=5, batch_size=128, verbose=1)
model.save('lstm_baseline.keras')
print('Model saved to lstm_baseline.keras')

In [ ]:
# Training results
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'])
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(history.history['accuracy'])
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

print(f'Final Loss: {history.history["loss"][-1]:.4f}')
print(f'Final Accuracy: {history.history["accuracy"][-1]:.4f}')

## 3. Text Generation (LSTM)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate_lstm_text(seed_text, n_words=20):
    result = seed_text
    for _ in range(n_words):
        token_list = tokenizer.texts_to_sequences([result])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)[0]
        # Get top 5 and choose randomly to avoid boring repetitions
        top_indices = np.argsort(predicted_probs)[-5:]
        predicted_id = np.random.choice(top_indices)
        output_word = ''
        for word, index in tokenizer.word_index.items():
            if index == predicted_id:
                output_word = word
                break
        result += ' ' + output_word
    return result

print('=== LSTM Text Generation ===')
seeds = ['адам баласы', 'ғылым мен білім', 'жас бала анадан']
for s in seeds:
    generated = generate_lstm_text(s, 20)
    print(f'\n[{s}] -> {generated}')

## 4. N-gram Language Model (Trigram / Markov Chain)

In [ ]:
from collections import defaultdict
import random
import pandas as pd
import string

df_train = pd.read_csv('data/train.csv')
texts = [str(t).lower().translate(str.maketrans('', '', string.punctuation)) for t in df_train['text']]
tokens = ' '.join(texts).split()

trigram_dict = defaultdict(list)
for i in range(len(tokens)-2):
    w1, w2, w3 = tokens[i], tokens[i+1], tokens[i+2]
    trigram_dict[(w1, w2)].append(w3)

def generate_ngram(seed, n_words=20):
    words = seed.lower().split()
    for _ in range(n_words):
        w1, w2 = words[-2], words[-1]
        if (w1, w2) in trigram_dict:
            next_word = random.choice(trigram_dict[(w1, w2)])
            words.append(next_word)
        else:
            break
    return ' '.join(words)

print('=== N-gram Text Generation ===')
for s in seeds:
    generated = generate_ngram(s, 20)
    print(f'\n[{s}] -> {generated}')

### Сравнение до / после аугментации (ТЗ 2.1)

Если в модуле 2 созданы `data/lstm_X_noaug.npy`, `data/lstm_y_noaug.npy`, `data/tokenizer_noaug.pkl`, повторите обучение: в ячейке загрузки данных подставьте эти файлы (и при генерации — `tokenizer_noaug.pkl`), сохраните модель под другим именем и сравните `loss` / качество генерации с вариантом после EDA.